# Лабораторная работа: полная конверсия ResNet-18 в SNN с частотным кодированием

## Цель работы

Самостоятельно реализовать конверсию предобученной `ResNet18` в спайковую сеть.

###План конверсии
1. Подготовка сети и калибровочного датасета.
2. Сбор статистик активаций на калибровочном датасете. Выбор активации отсечения.
3. Подготовка архитектуры. Слои BatchNorm сливаются с предшествующими свёртками
4. Замена нейронов. Слой ReLU заменяется на слой IF-нейронов с `reset-by-subtraction`;
5. Преобразование residual-блоков
6. Преобразование MaxPool

Этапы 1-4 реализованы в ноутбуках к лекции. Этапы 5-6 будут разобраны на лекции 6, но оставлены для самостоятельной реализации.

###Исследование
Исследовать качество полученной сети оценить для разных комбинаций параметров:
1. Длины временного окна $T=16,32,64,128,256$
2. Правила выбора активации отсечения: $max$, $p99$, $p99.9$

Для каждой комбинации оценить:
1. Качество классификацию ANN и SNN на тестовом фрагменте ImageNet. Не использовать для этого калибровочный датасет
2. Оценить переносимость признаков на любую выбранную задачу компьютерного зрения. Пример: сегментация, локализация.
3. Оценить число операций на одно изображение:
* MAC для ANN/SNN
* AC для ANN/SNN
* среднее число спайков на изображение
* среднее число спайков на один нейрон сети
* среднее число спайков на каждом слое сети

Ноутбук рассчитан на Google Colab с GPU. В режиме `FAST_MODE=True` используются небольшие подвыборки.

### Источники
* Презентации к лекциям 4,5
* Ноутбук с сетью VGG
* Статья по конверсии базовых свёрточных сетей
Rueckauer, B., Lungu, I. A., Hu, Y., Pfeiffer, M., & Liu, S. C. (2017). Conversion of continuous-valued deep networks to efficient event-driven networks for image classification. Frontiers in neuroscience, 11, 682.

* Статья по конверсии residual слоёв
Hu, Y., Tang, H., & Pan, G. (2021). Spiking deep residual networks. IEEE Transactions on Neural Networks and Learning Systems, 34(8), 5200-5205. URL:
https://arxiv.org/pdf/1805.01352


Можно использовать фреймворк SpikingJelly:
https://spikingjelly.readthedocs.io

## Теоретическая схема

Для ReLU-активации ANN $a_i^{(l)}\geq 0$ rate-конверсия использует число спайков IF-нейрона:

$$
\hat a_i^{(l)}(T)=\theta_l\frac{N_i^{(l)}(T)}{T},
$$

где $\theta_l$ — порог слоя, а $N_i^{(l)}(T)=\sum_{t=1}^{T}s_i^{(l)}[t]$.

Динамика IF-нейрона с мягким сбросом:

$$
u_i[t]=u_i[t-1]+I_i[t],
$$

$$
s_i[t]=H(u_i[t]-\theta_i),
$$

$$
u_i[t]\leftarrow u_i[t]-s_i[t]\theta_i.
$$

В отличие от `reset-to-zero`, вычитание порога сохраняет остаточный потенциал и уменьшает ошибку квантования частоты.

Выход IF-узла в программной модели записывается как $\theta s[t]$. На специализированном железе множитель $\theta$ можно заранее свернуть в веса следующего слоя, поэтому активный бинарный спайк инициирует прибавление предварительно масштабированного веса.

## 1. Установка и импорты

In [ ]:
# В Colab основные пакеты уже установлены.
#!pip -q install gdown pandas tqdm

In [ ]:
import copy
import json
import math
import random
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights
from torchvision.models.resnet import BasicBlock
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


## 2. Конфигурация эксперимента

In [ ]:
FAST_MODE = True
T_CHECKPOINTS = [16, 32, 64, 128, 256]

# Число объектов для калибровки порогов и оценки классификации.
CALIBRATION_SAMPLES = 128 if FAST_MODE else 1024
CLASSIFICATION_SAMPLES = 128 if FAST_MODE else None
CLASSIFICATION_BATCH_SIZE = 8 if torch.cuda.is_available() else 2

# Высокий перцентиль устойчивее абсолютного максимума.
THRESHOLD_PERCENTILE = 0.999
ACTIVATION_SAMPLE_LIMIT = 100_000

DATA_DIR = Path('/content/data') if Path('/content').exists() else Path('./data')
RESULTS_DIR = Path('/content/results') if Path('/content').exists() else Path('./results')
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('T checkpoints:', T_CHECKPOINTS)
print('FAST_MODE:', FAST_MODE)

T checkpoints: [16, 32, 64, 128, 256]
FAST_MODE: True


## 3. Фрагмент ImageNet: Imagenette

Ноутбук поддерживает фрагмент ImageNet в формате `ImageFolder`. По умолчанию используется ранее подготовленный ZIP-файл Imagenette на Google Drive.

Ожидаемая структура после распаковки:

```text
imagenette_subset/
  train/ или val/
    n01440764/
    n02102040/
    ...
```

Если файл уже находится на Google Drive, повторная загрузка не выполняется.

In [ ]:
DOWNLOAD_URL = 'https://drive.google.com/file/d/1JdnAT2kfg5CHby3DeL9q0fHkVdpluSrL/view?usp=sharing'
DRIVE_ZIP = Path('/content/drive/MyDrive/imagenette_subset/imagenette_subset.zip')
LOCAL_ZIP = DATA_DIR / 'imagenette_subset.zip'
EXTRACT_DIR = DATA_DIR / 'imagenette_subset'

def prepare_imagenette_zip(download=True):
    """Монтирует Drive, при необходимости загружает ZIP и распаковывает его."""
    if Path('/content').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
        except Exception as exc:
            print('Drive не смонтирован:', exc)

    zip_path = DRIVE_ZIP if DRIVE_ZIP.exists() else LOCAL_ZIP
    if not zip_path.exists() and download:
        import gdown
        LOCAL_ZIP.parent.mkdir(parents=True, exist_ok=True)
        gdown.download(url=DOWNLOAD_URL, output=str(LOCAL_ZIP), fuzzy=True, quiet=False)
        zip_path = LOCAL_ZIP

    if not zip_path.exists():
        raise FileNotFoundError(
            'ZIP не найден. Поместите его в DRIVE_ZIP или измените DOWNLOAD_URL.'
        )

    marker = EXTRACT_DIR / '.extracted'
    if not marker.exists():
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(EXTRACT_DIR)
        marker.touch()
    return EXTRACT_DIR

def find_imagefolder_root(root):
    """Ищет каталог val/train, содержащий папки классов ImageNet."""
    root = Path(root)
    candidates = []
    for path in [root, *root.rglob('*')]:
        if not path.is_dir():
            continue
        dirs = [p.name for p in path.iterdir() if p.is_dir()]
        if any(name.startswith('n') and len(name) == 9 for name in dirs):
            score = 2 if path.name == 'val' else 1 if path.name == 'train' else 0
            candidates.append((score, path))
    if not candidates:
        raise RuntimeError(f'Не найден ImageFolder внутри {root}')
    return sorted(candidates, key=lambda item: item[0], reverse=True)[0][1]

imagenette_root = find_imagefolder_root(prepare_imagenette_zip())
print('ImageFolder root:', imagenette_root)

Mounted at /content/drive


Downloading...
From (original): https://drive.google.com/uc?id=1JdnAT2kfg5CHby3DeL9q0fHkVdpluSrL
From (redirected): https://drive.google.com/uc?id=1JdnAT2kfg5CHby3DeL9q0fHkVdpluSrL&confirm=t&uuid=d6b6b4b4-8bc3-4337-a3ee-e5be277d1211
To: /content/data/imagenette_subset.zip
100%|██████████| 143M/143M [00:05<00:00, 28.2MB/s]


ImageFolder root: /content/data/imagenette_subset/imagenette_10k_subset


In [ ]:
weights = ResNet18_Weights.DEFAULT
imagenet_transform = weights.transforms()

# Соответствие десяти WNID Imagenette исходным индексам ImageNet-1k.
IMAGENETTE_TO_IMAGENET = {
    'n01440764': 0,
    'n02102040': 217,
    'n02979186': 482,
    'n03000684': 491,
    'n03028079': 497,
    'n03394916': 566,
    'n03417042': 569,
    'n03425413': 571,
    'n03445777': 574,
    'n03888257': 701,
}

class RemapImageNetTargets(Dataset):
    def __init__(self, image_folder, mapping):
        self.base = image_folder
        self.mapping = mapping
        self.class_to_imagenet = {
            local_idx: mapping[class_name]
            for class_name, local_idx in image_folder.class_to_idx.items()
        }

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        image, local_target = self.base[index]
        return image, self.class_to_imagenet[local_target]

base_imagenette = datasets.ImageFolder(imagenette_root, transform=imagenet_transform)
missing = set(base_imagenette.classes) - set(IMAGENETTE_TO_IMAGENET)
if missing:
    raise ValueError(f'Неизвестные WNID: {sorted(missing)}')

imagenette = RemapImageNetTargets(base_imagenette, IMAGENETTE_TO_IMAGENET)
if CLASSIFICATION_SAMPLES is not None:
    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(len(imagenette), generator=generator)[:CLASSIFICATION_SAMPLES]
    imagenette_eval = Subset(imagenette, indices.tolist())
else:
    imagenette_eval = imagenette

classification_loader = DataLoader(
    imagenette_eval,
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

calibration_indices = list(range(min(CALIBRATION_SAMPLES, len(imagenette))))
calibration_loader = DataLoader(
    Subset(imagenette, calibration_indices),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print('classes:', base_imagenette.classes)
print('evaluation images:', len(imagenette_eval))

classes: ['n01440764', 'n02102040', 'n02979186', 'n03028079', 'n03394916', 'n03425413', 'n03888257']
evaluation images: 128


## 4. Исходная ANN и базовое качество

In [ ]:
@torch.inference_mode()
def classification_accuracy(model, loader, max_batches=None):
    model.eval()
    correct = 0
    total = 0
    for batch_idx, (images, targets) in enumerate(tqdm(loader, leave=False)):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images = images.to(device)
        targets = targets.to(device)
        logits = model(images)
        correct += (logits.argmax(1) == targets).sum().item()
        total += targets.numel()
    return correct / max(total, 1)

ann = models.resnet18(weights=weights).eval().to(device)
ann_accuracy = classification_accuracy(ann, classification_loader)
print(f'ANN accuracy: {ann_accuracy:.4f}')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 202MB/s]


  0%|          | 0/16 [00:00<?, ?it/s]

ANN accuracy: 0.7109


## 5. Подсчёт MAC исходной ANN

Для свёртки без учёта bias:

$$
N_{MAC}=H_{out}W_{out}C_{out}\frac{C_{in}}{g}K_hK_w.
$$

Для полносвязного слоя:

$$
N_{MAC}=N_{in}N_{out}.
$$

После BatchNorm folding операции BatchNorm отдельно не учитываются.

In [ ]:
def dense_ops(module, x, y):
    batch = x.shape[0]
    if isinstance(module, nn.Conv2d):
        out_h, out_w = y.shape[-2:]
        kernel_h, kernel_w = module.kernel_size
        per_sample = (
            out_h * out_w * module.out_channels
            * (module.in_channels // module.groups)
            * kernel_h * kernel_w
        )
    elif isinstance(module, nn.Linear):
        per_sample = module.in_features * module.out_features
    else:
        return 0
    return int(batch * per_sample)

class ANNMacCounter:
    def __init__(self, model):
        self.total = 0
        self.handles = []
        for module in model.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                self.handles.append(module.register_forward_hook(self._hook))

    def _hook(self, module, inputs, output):
        self.total += dense_ops(module, inputs[0], output)

    def close(self):
        for handle in self.handles:
            handle.remove()

with torch.inference_mode():
    sample_images, _ = next(iter(classification_loader))
    sample_images = sample_images[:1].to(device)
    counter = ANNMacCounter(ann)
    _ = ann(sample_images)
    ann_macs_per_image = counter.total
    counter.close()

print(f'ANN MAC/image: {ann_macs_per_image:,}')

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a1acbf72e0>
Traceback (most recent call last):
Exception ignored in:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x78a1acbf72e0>
    Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
self._shutdown_workers()    
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
if w.is_alive():
      File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
if w.is_alive():    
assert self._parent_pid == os.getpid(), 'can only test a child process'
  File "/usr/lib/python3.13/multiprocessing/process.py", line 1

ANN MAC/image: 1,814,073,344


## 6. BatchNorm folding

Для свёртки $y=Wx+b$ и BatchNorm в режиме inference:

$$
\operatorname{BN}(y)=\gamma\frac{y-\mu}{\sqrt{\sigma^2+\varepsilon}}+\beta.
$$

Эквивалентные параметры:

$$
W'=\frac{\gamma}{\sqrt{\sigma^2+\varepsilon}}W,
$$

$$
b'=\frac{\gamma}{\sqrt{\sigma^2+\varepsilon}}(b-\mu)+\beta.
$$

После преобразования BatchNorm заменяется на `Identity`.

In [ ]:
# Реализовать

## 7. Калибровка порогов IF-нейронов

Порог слоя оценивается по распределению положительных ReLU-активаций:

$$
\theta_l=Q_p\left(a^{(l)}\right),
$$

где $Q_p$ — перцентиль, например $p=0.999$.

Абсолютный максимум чувствителен к единичным выбросам. Слишком низкий порог приводит к насыщению $N(T)=T$, слишком высокий — к малому числу спайков и большой ошибке квантования.

In [ ]:
# реализовать

## 8. IF-нейрон с soft reset

`IFNode` хранит мембранный потенциал между временными шагами, но состояние обязательно обнуляется перед новым batch. Поле `spike_count` нужно только для диагностики и не участвует в вычислении выхода.

In [ ]:
class IFNode(nn.Module):
    def __init__(self, threshold, name):
        super().__init__()
        self.register_buffer('threshold', torch.tensor(float(threshold)))
        self.name = name
        self.membrane = None
        self.spike_count = 0

    def reset_state(self):
        self.membrane = None
        self.spike_count = 0

    def forward(self, current):
        if self.membrane is None or self.membrane.shape != current.shape:
            self.membrane = torch.zeros_like(current)
        self.membrane = self.membrane + current
        spike = self.membrane.ge(self.threshold).to(current.dtype)
        self.membrane = self.membrane - spike * self.threshold
        self.spike_count += int(spike.count_nonzero().item())
        return spike * self.threshold

## 9. Конверсия MaxPool блока
Блок MaxPool должен динамически оценить частоту потока спайков $\hat{r_i}$ и выбрать победителя за подвижный временной интервал.



## 10. Конверсия residual-блока

ANN-блок:

$$
y=\operatorname{ReLU}\left(F(x)+x\right),
$$

В SNN основная и residual-ветви формируют ток одного выходного IF-нейрона:

$$
I_y[t]=F_s(x[t])+x[t],
$$

$$
y[t]=\operatorname{IF}_{\theta_2}(I_y[t]).
$$


In [ ]:
class SNNOperationCounter:
    """Оценка MAC и AC во время фактического forward.

    Ненулевой элемент входного spike tensor считается активным событием.
    Для Conv/Linear AC оцениваются как dense_ops * input_density.
    """
    def __init__(self):
        self.mac = 0.0
        self.ac = 0.0
        self.residual_ac = 0.0
        self.enabled = False
        self.analog_modules = set()
        self.handles = []

    def attach(self, model, analog_modules=()):
        self.analog_modules = {id(module) for module in analog_modules}
        for module in model.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                self.handles.append(module.register_forward_hook(self._hook))

    def _hook(self, module, inputs, output):
        if not self.enabled:
            return
        x = inputs[0]
        dense = dense_ops(module, x, output)
        if id(module) in self.analog_modules:
            self.mac += dense
        else:
            density = float(x.count_nonzero().item()) / max(x.numel(), 1)
            self.ac += dense * density

    def add_residual(self, main, identity):
        if self.enabled:
            self.residual_ac += main.count_nonzero().item() + identity.count_nonzero().item()

    def reset(self):
        self.mac = self.ac = self.residual_ac = 0.0

    def snapshot(self):
        return {
            'mac': self.mac,
            'synaptic_ac': self.ac,
            'residual_ac': self.residual_ac,
            'total_ac': self.ac + self.residual_ac,
        }

    def close(self):
        for handle in self.handles:
            handle.remove()

class SpikingBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, ann_block, threshold1, threshold2, name, op_counter=None):
        #реализовать

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.if1(self.conv1(x))
        out = self.conv2(out)
        if self.op_counter is not None:
            self.op_counter.add_residual(out, identity)
        out = self.if2(out + identity)
        return out

## 11. Полная SNN ResNet-18

Первый слой по умолчанию получает аналоговое нормализованное изображение. Его свёртка вычисляется один раз, а постоянный ток подаётся на первый IF-нейрон в каждом такте. Это стандартный практический компромисс: скрытые представления являются спайковыми, а дорогая первая свёртка не повторяется $T$ раз.

Опциональный режим `signed_poisson` генерирует знаковые входные события для нормализованных пикселей, но обычно имеет большую дисперсию и работает медленнее.

In [ ]:
# Реализовать

### Самопроверка residual-блока

Проверьте, что:

- в сети нет `BatchNorm2d`;
- число IF-узлов равно числу ReLU-сайтов;
- у каждого BasicBlock существуют отдельные `if1` и `if2`;
- downsample содержит только свёртку.

In [ ]:
assert not any(isinstance(module, nn.BatchNorm2d) for module in snn.modules())
if_nodes = [module for module in snn.modules() if isinstance(module, IFNode)]
print('IF nodes:', len(if_nodes))
print('IF names:', [node.name for node in if_nodes])
assert len(if_nodes) == 17  # 1 входная ReLU + 2 ReLU × 8 BasicBlock

IF nodes: 17
IF names: ['relu', 'layer1.0.relu1', 'layer1.0.relu2', 'layer1.1.relu1', 'layer1.1.relu2', 'layer2.0.relu1', 'layer2.0.relu2', 'layer2.1.relu1', 'layer2.1.relu2', 'layer3.0.relu1', 'layer3.0.relu2', 'layer3.1.relu1', 'layer3.1.relu2', 'layer4.0.relu1', 'layer4.0.relu2', 'layer4.1.relu1', 'layer4.1.relu2']


## 12. Классификация Imagenette при разных $T$

In [ ]:
@torch.inference_mode()
def evaluate_snn_checkpoints(model, loader, checkpoints):
    totals = {
        t: {
            'correct': 0,
            'samples': 0,
            'spikes': 0,
            'mac': 0.0,
            'synaptic_ac': 0.0,
            'residual_ac': 0.0,
        }
        for t in checkpoints
    }

    for images, targets in tqdm(loader, desc='SNN classification'):
        images = images.to(device)
        targets = targets.to(device)
        batch_outputs = model.forward_checkpoints(images, checkpoints)

        for t, output in batch_outputs.items():
            row = totals[t]
            row['correct'] += (output['logits'].argmax(1) == targets).sum().item()
            row['samples'] += targets.numel()
            row['spikes'] += output['spikes']
            row['mac'] += output['ops']['mac']
            row['synaptic_ac'] += output['ops']['synaptic_ac']
            row['residual_ac'] += output['ops']['residual_ac']

    rows = []
    for t in checkpoints:
        row = totals[t]
        n = max(row['samples'], 1)
        total_ac = row['synaptic_ac'] + row['residual_ac']
        rows.append({
            'T': t,
            'accuracy': row['correct'] / n,
            'accuracy_drop': ann_accuracy - row['correct'] / n,
            'spikes_per_image': row['spikes'] / n,
            'MAC_per_image': row['mac'] / n,
            'synaptic_AC_per_image': row['synaptic_ac'] / n,
            'residual_AC_per_image': row['residual_ac'] / n,
            'total_AC_per_image': total_ac / n,
            'AC_to_ANN_MAC': (total_ac / n) / ann_macs_per_image,
        })
    return pd.DataFrame(rows)

classification_results = evaluate_snn_checkpoints(
    snn, classification_loader, T_CHECKPOINTS
)
classification_results

In [ ]:
classification_results.to_csv(
    RESULTS_DIR / 'resnet18_rate_conversion_classification.csv', index=False
)

ax = classification_results.plot(
    x='T', y=['accuracy', 'AC_to_ANN_MAC'], marker='o', grid=True,
    title='Качество и относительное число AC'
)
ax.figure.tight_layout()

## 13. Адаптация под другую задачу
Полученный энкодер ResNet можно использовать для другой задачи компьютерного зрения. Выберите задачу и датасет. Например, сегментация, детекция или локализация.

Замените в нужной архитектуре энкодер на спайковый ResNet и оцените качество.

## 14. Сводная таблица

При интерпретации результатов необходимо различать:

- **MAC ANN** — умножение с накоплением для плотной сети;
- **AC/SynOps SNN** — прибавление веса при входном событии;
- **residual AC** — сложения двух ветвей residual-блока;
- **spikes** — число событий, а не число синаптических операций;

Один спайк может инициировать много SynOps, поскольку связан со многими нейронами следующего слоя.

## 15. Задания для самостоятельной работы

### Обязательная часть

1. Запустите калибровку с $p\in\{0.99,0.999,1.0\}$ и объясните влияние порога на точность и число спайков.
2. Заполните таблицу для $T=16,32,64,128,256$.
3. Найдите минимальный $T$, при котором падение accuracy относительно ANN не превышает 2 процентных пунктов.
4. Сравните `spikes_per_image` и `total_AC_per_image`. В чём отличие?

### Вопросы к отчёту

- Какие ошибки конверсии уменьшаются при росте $T$?
- Почему BatchNorm необходимо свернуть до конверсии?
- Почему shortcut и основная ветвь должны иметь согласованный масштаб?
- Почему уменьшение числа спайков не гарантирует уменьшение энергии?
- Какие операции этой реализации не вошли в MAC/AC-модель?

## 16. Формат отчёта

Отчёт должен содержать:

1. краткое описание алгоритма конверсии;
2. проверку эквивалентности ANN до и после BatchNorm folding;
3. таблицу accuracy, MAC, AC и числа спайков для всех $T$;
4. график accuracy–$T$ и AC–$T$;
5. таблицу IoU/Dice для задачи сегментации;
6. примеры исходного изображения, разметки, ANN- и SNN-предсказания;
7. обсуждение ограничений принятой модели стоимости;
8. вывод о рекомендуемом $T$.